In [1]:
import pandas as pd
import sys
sys.path.append('..')
from pipeline.preprocessors.revised_missing import RevisedMissingValueHandler
from pipeline.preprocessors.categorical import CategoricalEncoder

In [2]:
df = pd.read_csv('../../data/diabetes.csv')

impossibles = ['Glucose', 'BloodPressure', 'SkinThickness', 'BMI']

missing_handler = RevisedMissingValueHandler(
    drop_cols=['Insulin'],
    impute_cols=impossibles,
    zero_cols=impossibles
)

df_clean = missing_handler.fit_transform(df)

## BMI Categories: NHS thresholds
BMI categories defined using official NHS guidelines:
- Underweight: < 18.5
- Healthy weight: 18.5 - 24.9
- Overweight: 25 - 29.9
- Obese: ≥ 30

Source: https://www.diabetes.co.uk/bmi.html

In [3]:
df_clean['BMICategory'] = pd.cut(df_clean['BMI'],
                                  bins=[0, 18.5, 24.9, 29.9, 100],
                                  labels=['Underweight', 'Healthy', 'Overweight', 'Obese'])

In [4]:
encoder = CategoricalEncoder(
    nominal_cols=[],
    ordinal_cols={'BMICategory': ['Underweight', 'Healthy', 'Overweight', 'Obese']}
)

df_encoded = encoder.fit_transform(df_clean)
df_encoded.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,BMI,DiabetesPedigreeFunction,Age,Outcome,BMICategory
0,6,148.0,72.0,35.00000,33.6,0.627,50,1,3.0
1,1,85.0,66.0,29.00000,26.6,0.351,31,0,2.0
2,8,183.0,64.0,29.15342,23.3,0.672,32,1,1.0
3,1,89.0,66.0,23.00000,28.1,0.167,21,0,2.0
4,0,137.0,40.0,35.00000,43.1,2.288,33,1,3.0


In [5]:
df_encoded['BMICategory'].value_counts()

BMICategory
3.0    459
2.0    173
1.0     97
0.0      4
Name: count, dtype: int64

In [11]:
overweight_pct = (df_encoded['BMICategory'].astype(float) >= 2.0).sum() /df_encoded.shape[0] * 100
print(overweight_pct)

86.22100954979535


## BMICategory encoding result
- Ordinal encoding correctly preserves NHS BMI category order.
- Distribution is clinically consistent with a diabetes dataset.
- 86% of patients fall in Overweight or Obese categories.